In [ ]:
import sys
import vega
import os
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import ast
from joblib import Parallel, delayed
from tqdm_joblib import tqdm_joblib
from tqdm import tqdm

REPO_ROOT = Path("../../../../..").resolve()
VEGA_REPRO = REPO_ROOT / "models_reproductibility" / "vega-reproducibility"
sys.path.append(str(VEGA_REPRO / "src"))
import vanilla_vae
import train_vanilla_vae_suppFig1
import utils
from utils import *
from learning_utils import *
from vanilla_vae import VanillaVAE
from vega_model import VEGA

sys.path.append(str(REPO_ROOT / "utils" / "utils_evaluation"))
import vega_utils
from vega_utils import *

sys.path.append(str(REPO_ROOT / "utils" / "utils_interpretability"))
import utils_vega_perturbation
from utils_vega_perturbation import *
import utils_compute_scores2 
from utils_compute_scores2 import *


In [ ]:
name_dataset = "CHOL_GSE138709"
type_data = 'tisch'


In [ ]:
name_model = 'Vega'
select_hvg = True
n_top_genes = 2000
random_seed = 42
train_size = 0.9
preprocess = True


In [ ]:
pathway_file = str(VEGA_REPRO / "data" / "reactomes.gmt")
path_data = str(REPO_ROOT / "datasets" / "tischdb") + "/"

#Load data
if type_data == 'tisch':
    metadata = pd.read_csv(path_data + f'{name_dataset}_CellMetainfo_table.tsv', sep="\t")
    adata = sc.read_10x_h5(path_data + f'{name_dataset}_expression.h5', genome='GRCh38', gex_only=False)
    adata.obs = metadata
    column_labels_name = 'Celltype (major-lineage)' 
if name_dataset == 'Kang_PBMC':
    adata = sc.read(str(REPO_ROOT / "datasets" / "VEGA" / "Kang PBMC" / "train_pbmc.h5ad"))
    column_labels_name = 'cell_type' 

#prepare data
adata, adata_train, adata_test, train_data, test_data, pathway_dict, pathway_mask = create_vega_training_data(name_model, preprocess, select_hvg, n_top_genes, random_seed, train_size, column_labels_name, adata, pathway_file)
adata, adata_train, adata_val, train_data, val_data, pathway_dict, pathway_mask = create_vega_training_data(name_model, False, select_hvg, n_top_genes, random_seed, train_size, column_labels_name, adata_train, pathway_file)
adata, pathway_dict, pathway_mask, list_pathways, df_genespathways, overlap_matrix = access_data_vega(None, adata, pathway_file)


In [ ]:
overlap_matrix


In [ ]:
name_model = 'Vega'
name_activation_column = 'original neuron activation'
name_overlap_compared_pathway = 'Compared Pathway'
name_overlap_column = 'Overlap Proportion'
#activ_threshold = 0.5
overlap_threshold = 0.5
split = 'test'


In [ ]:
OUTPUT_ROOT = REPO_ROOT / "outputs"
path_to_save_reduction_scores = f'/home/data/{name_model}/{name_dataset}/reduction_scores/'
path_to_save_results = str(OUTPUT_ROOT / name_model / f"interpretability results/{name_dataset}/all probas perturbations 2") + "/"


In [ ]:
n_cpus = os.cpu_count()
n_cpus


In [ ]:
OUTPUT_ROOT = REPO_ROOT / "outputs"
results_list = []
final_df_list = []

with tqdm_joblib(total=len(list_pathways[:-1])):
    output = Parallel(n_jobs=n_cpus-16)(
        delayed(compute_metrics_for_pathway)(
            'vega',
            name_dataset,
            pathway_perturbated,
            list_pathways[:-1],  # as in your original call
            path_to_save_reduction_scores,
            split,
            overlap_matrix,
            name_activation_column,
            name_overlap_compared_pathway,
            name_overlap_column,
            #activ_threshold,
            overlap_threshold,
            path_to_save_results
        )
        for pathway_perturbated in list_pathways[:-1]
    )

# Unpack results
results_list, final_df_list = zip(*output)
combined_df = pd.concat(results_list, ignore_index=True)
combined_df.to_csv(str(OUTPUT_ROOT / name_model / f"interpretability results/{name_dataset}/all_probas_final_df_2.csv") + "/")


In [ ]:
combined_df


In [ ]:
combined_df.mean()
